# OOP Week 4 -- Analyzer Component

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-3
**Focus:** analysis as an object, stable results schema, composable analyzers

---

## Learning Objectives

1. Build an Analyzer class with a stable output schema
2. Understand why analysis results should follow a fixed structure
3. Create multiple analyzer types (mean, std, event detection)
4. Compose analyzers to produce a combined report
5. Add threshold-based event detection

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Setup: Dataset from previous weeks

In [ ]:
class Dataset:
    def __init__(self, rows, source_path="unknown"):
        self.rows = rows
        self.source_path = source_path
        self.n_rows = len(rows)
        self.columns = list(rows[0].keys()) if rows else []
    def get_column(self, name):
        return [row.get(name) for row in self.rows]
    def __len__(self):
        return self.n_rows
    def __str__(self):
        return "Dataset(" + str(self.n_rows) + " rows)"

# Clean sample data
clean_data = Dataset([
    {"id": 1, "value": 25.0, "status": "ok"},
    {"id": 2, "value": 30.0, "status": "ok"},
    {"id": 3, "value": 88.0, "status": "warning"},
    {"id": 4, "value": 42.0, "status": "ok"},
    {"id": 5, "value": 67.0, "status": "ok"},
    {"id": 6, "value": 15.0, "status": "ok"},
], "sensors.csv")
print("Clean data:", clean_data)

**Expected Output:**
```
Clean data: Dataset(6 rows)
```

---
## Section 1: Why Wrap Analysis in a Class?

In your v2 pipeline, analysis was probably a function that returned a dictionary. That works, but:

1. **No guaranteed structure** -- different functions return different keys
2. **No metadata** -- when was the analysis run? On what data?
3. **No composition** -- hard to combine multiple analyses
4. **No state** -- cannot inspect what happened after the fact

An Analyzer class solves all of these.

---
### Procedural vs OOP: Data Analysis

With OOP, every analyzer follows the same interface: call `.analyze(dataset)` and get back a dictionary with a guaranteed structure. This makes it easy to combine, test, and swap analyzers.

**Procedural approach (what you did in CP1/CP2):**

In [ ]:
# PROCEDURAL
def analyze_mean(data, column):
    vals = [r[column] for r in data if isinstance(r.get(column), (int, float))]
    return {"mean": sum(vals) / len(vals)} if vals else {}

def analyze_std(data, column):
    vals = [r[column] for r in data if isinstance(r.get(column), (int, float))]
    if not vals:
        return {}
    m = sum(vals) / len(vals)
    return {"std": (sum((x-m)**2 for x in vals) / len(vals)) ** 0.5}

# Different functions, different return shapes, no guarantees
r1 = analyze_mean([{"value": 10}, {"value": 20}], "value")
r2 = analyze_std([{"value": 10}, {"value": 20}], "value")
print(r1, r2)

**OOP approach (what we are learning now):**

In [ ]:
# OOP -- consistent interface
# (defined below)
print("All analyzers will have: .analyze(dataset) -> dict")
print("All results will have: column, count, result keys")

---
## Section 2: The Analyzer Class

In [ ]:
class Analyzer:
    """Analyzes a Dataset and produces structured results."""

    def __init__(self, config):
        self.config = config
        self.results = {}

    def analyze(self, dataset):
        """Run analysis and return results dict."""
        col = self.config.get("value_column", "value")
        values = []
        for row in dataset.rows:
            val = row.get(col)
            if isinstance(val, (int, float)):
                values.append(val)

        if not values:
            self.results = {"analysis_summary": {"count": 0}}
            return self.results

        n = len(values)
        mean_val = sum(values) / n
        sorted_vals = sorted(values)
        median_val = sorted_vals[n // 2]
        variance = sum((x - mean_val) ** 2 for x in values) / n
        std_val = variance ** 0.5

        self.results = {
            "analysis_summary": {
                "column": col,
                "count": n,
                "mean": round(mean_val, 4),
                "median": round(median_val, 4),
                "std": round(std_val, 4),
                "min": round(min(values), 4),
                "max": round(max(values), 4),
            }
        }

        # Optional: event detection
        threshold = self.config.get("threshold")
        if threshold is not None:
            events = sum(1 for v in values if v > threshold)
            self.results["analysis_summary"]["events_above_threshold"] = events
            self.results["analysis_summary"]["threshold"] = threshold

        return self.results


# Test
analyzer = Analyzer({"value_column": "value", "threshold": 50})
results = analyzer.analyze(clean_data)

import json
print(json.dumps(results, indent=2))

**Expected Output:**
```
{
  "analysis_summary": {
    "column": "value",
    "count": 6,
    "mean": 44.5,
    "median": 42.0,
    "std": 25.1330,
    "min": 15.0,
    "max": 88.0,
    "events_above_threshold": 2,
    "threshold": 50
  }
}
```

---
## Section 3: Composable Analyzers

Like cleaners, we can create specialized analyzers and combine them.

In [ ]:
class AnalyzerBase:
    """Base class for all analyzers."""
    def analyze(self, values):
        raise NotImplementedError("Subclasses must implement analyze()")

class MeanAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        return {"mean": round(sum(values) / len(values), 4)}

class StdAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        m = sum(values) / len(values)
        var = sum((x - m) ** 2 for x in values) / len(values)
        return {"std": round(var ** 0.5, 4)}

class EventAnalyzer(AnalyzerBase):
    def __init__(self, threshold):
        self.threshold = threshold
    def analyze(self, values):
        events = sum(1 for v in values if v > self.threshold)
        return {"events_above": events, "threshold": self.threshold}

class RangeAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        return {"min": min(values), "max": max(values), "range": max(values) - min(values)}


# Compose and run all
values = clean_data.get_column("value")
values = [v for v in values if isinstance(v, (int, float))]

analyzers = [
    MeanAnalyzer(),
    StdAnalyzer(),
    EventAnalyzer(threshold=50),
    RangeAnalyzer(),
]

combined = {}
for a in analyzers:
    result = a.analyze(values)
    combined.update(result)
    print(type(a).__name__ + ":", result)

print()
print("Combined:", combined)

**Expected Output:**
```
MeanAnalyzer: {'mean': 44.5}
StdAnalyzer: {'std': 25.133}
EventAnalyzer: {'events_above': 2, 'threshold': 50}
RangeAnalyzer: {'min': 15.0, 'max': 88.0, 'range': 73.0}

Combined: {'mean': 44.5, 'std': 25.133, 'events_above': 2, 'threshold': 50, 'min': 15.0, 'max': 88.0, 'range': 73.0}
```

---
### Design Decision: Stable Results Schema

Notice that every analyzer returns a dictionary with predictable keys. This is crucial because:

1. **The Reporter** can always find the data it needs
2. **Tests** can assert specific keys exist
3. **New analyzers** can be added without breaking existing code
4. **JSON export** just works

A stable schema is a **contract** between components. Break it and everything downstream breaks.

---
### Try It!

Create a `PercentileAnalyzer(AnalyzerBase)` that computes the 25th, 50th, and 75th percentiles. Add it to the list and run again.

In [ ]:
# YOUR CODE HERE


---
### Common Mistake: Not handling empty input

The code below has a bug. Can you spot it before reading the fix?

In [ ]:
class BadAnalyzer:
    def analyze(self, values):
        # BUG: crashes on empty list!
        return {"mean": sum(values) / len(values)}

try:
    BadAnalyzer().analyze([])
except ZeroDivisionError as e:
    print("ERROR:", e)

**What goes wrong:** Always handle edge cases first (empty list, None, etc). This is called a **guard clause** pattern.

**The fix:**

In [ ]:
class SafeAnalyzer:
    def analyze(self, values):
        if not values:          # FIXED: guard clause
            return {"mean": 0}  # or return {}
        return {"mean": sum(values) / len(values)}

print(SafeAnalyzer().analyze([]))  # safe!
print(SafeAnalyzer().analyze([10, 20]))  # normal case

---
## Section 4: More Analyzer Types

### Example: CountAnalyzer

In [ ]:
class CountAnalyzer(AnalyzerBase):
    """Counts values meeting various criteria."""
    def analyze(self, values):
        if not values:
            return {"count": 0}
        positives = sum(1 for v in values if v > 0)
        negatives = sum(1 for v in values if v < 0)
        zeros = sum(1 for v in values if v == 0)
        return {
            "count": len(values),
            "positives": positives,
            "negatives": negatives,
            "zeros": zeros,
        }

ca = CountAnalyzer()
test_vals = [10, -5, 0, 30, -2, 0, 15]
print("Count analysis:", ca.analyze(test_vals))

**Expected Output:**
```
Count analysis: {'count': 7, 'positives': 3, 'negatives': 2, 'zeros': 2}
```

### Example: TrendAnalyzer

In [ ]:
class TrendAnalyzer(AnalyzerBase):
    """Detects if values are trending up, down, or flat."""
    def analyze(self, values):
        if len(values) < 2:
            return {"trend": "insufficient_data"}
        mid = len(values) // 2
        first_half = sum(values[:mid]) / mid
        second_half = sum(values[mid:]) / (len(values) - mid)
        diff = second_half - first_half
        if abs(diff) < 1.0:
            trend = "flat"
        elif diff > 0:
            trend = "increasing"
        else:
            trend = "decreasing"
        return {
            "trend": trend,
            "first_half_mean": round(first_half, 2),
            "second_half_mean": round(second_half, 2),
        }

ta = TrendAnalyzer()
print("Increasing:", ta.analyze([10, 12, 15, 20, 25, 30]))
print("Decreasing:", ta.analyze([30, 25, 20, 15, 12, 10]))
print("Flat:      ", ta.analyze([20, 20, 20, 20, 20, 20]))

**Expected Output:**
```
Increasing: {'trend': 'increasing', 'first_half_mean': 12.33, 'second_half_mean': 25.0}
Decreasing: {'trend': 'decreasing', 'first_half_mean': 25.0, 'second_half_mean': 12.33}
Flat:       {'trend': 'flat', 'first_half_mean': 20.0, 'second_half_mean': 20.0}
```

---
## Section 5: Running All Analyzers Together

In [ ]:
def run_all_analyzers(analyzers, values):
    """Run all analyzers and merge results."""
    combined = {}
    for a in analyzers:
        name = type(a).__name__
        result = a.analyze(values)
        combined[name] = result
        print(name + ":", result)
    return combined

all_analyzers = [
    MeanAnalyzer(),
    StdAnalyzer(),
    EventAnalyzer(50),
    RangeAnalyzer(),
    CountAnalyzer(),
    TrendAnalyzer(),
]

values = [15, 25, 30, 42, 67, 88]
print("=== Running 6 analyzers ===")
results = run_all_analyzers(all_analyzers, values)

**Expected Output:**
```
=== Running 6 analyzers ===
MeanAnalyzer: {'mean': 44.5}
StdAnalyzer: {'std': 25.133}
EventAnalyzer: {'events_above': 2, 'threshold': 50}
RangeAnalyzer: {'min': 15, 'max': 88, 'range': 73}
CountAnalyzer: {'count': 6, 'positives': 6, 'negatives': 0, 'zeros': 0}
TrendAnalyzer: {'trend': 'increasing', ...}
```

---
### Debugging Tip: KeyError when accessing results

If your analyzer returns `{"mean": 20}` but your reporter tries `results["avg"]`, you get a KeyError. This is why a **stable schema** matters -- everyone agrees on the key names.

**Fix:** Document the exact keys each analyzer returns. Write tests that assert the expected keys exist.

---
### Try It!

Create a `PercentileAnalyzer(AnalyzerBase)` that takes a list of percentiles (e.g., [25, 50, 75]) and returns them as `{"p25": ..., "p50": ..., "p75": ...}`. Test with the sample data.

In [ ]:
# YOUR CODE HERE


---
### Try It!

Create a `HistogramAnalyzer(AnalyzerBase)` that bins values into ranges and counts them. For example, with bins of width 20:
`{"0-20": 1, "20-40": 2, "40-60": 1, "60-80": 1, "80-100": 1}`

In [ ]:
# YOUR CODE HERE


---
## Build from Scratch Exercise

This exercise tests whether you truly understand this week's concepts. Complete it without looking at the examples above.

In [ ]:
# BUILD FROM SCRATCH:
# Build an AnalyzerSuite that runs GPA analysis: MeanGPA, MedianGPA, HighestGPA, LowestGPA. Each inherits from AnalyzerBase. Combine results into one dict.

# YOUR CODE HERE


In [ ]:
# TEST your build-from-scratch code:

# YOUR TESTS HERE


---
## Connect the Dots

How does this week's concept connect to previous weeks?

In [ ]:
# How is the Analyzer similar to the Cleaner (Week 3)? How is it different?

# YOUR ANSWER (as comments or code):


---
## Real-World Spotting

OOP patterns are everywhere in real software. Can you spot them?

In [ ]:
# Think about a fitness tracker. What analyses would it run on step data? Design 3 analyzer classes.

# YOUR ANSWER:


---
## Diagram It

Draw an ASCII class diagram for the main classes from this week. Include:
- Class names
- Key attributes
- Key methods
- Relationships (has-a, is-a)

In [ ]:
# Draw your ASCII diagram here:
# +------------------+
# |   ClassName      |
# +------------------+
# | - attribute      |
# +------------------+
# | + method()       |
# +------------------+

# YOUR DIAGRAM:


---
## Key Vocabulary

| Term | Definition |
|------|------------|
| **Schema** | The expected structure of data (column names, types) |
| **Stable schema** | A schema that does not change between versions |
| **Guard clause** | An early return that handles edge cases |
| **NotImplementedError** | Raised when a base class method must be overridden |

---
## Recap Exercise

Without looking at the code above, try to:

In [ ]:
# 1. Write one class from this week FROM MEMORY
#    (it does not need to be perfect)

# YOUR CODE HERE


# 2. Create an instance and call at least one method

# YOUR CODE HERE


# 3. Write one test for your class

# YOUR CODE HERE


---
## What to Review Before Next Week

Before the next session, make sure you can:

1. Explain this week's main concept in your own words
2. Write a simple example from memory
3. Identify this pattern in existing code
4. Explain WHY this pattern is useful (not just HOW)

---
## Mini-Quiz

In [ ]:
# Q1: Why do we wrap analysis in a class instead of a plain function?
# Answer: 

# Q2: What does 'stable schema' mean?
# Answer: 

# Q3: What does AnalyzerBase.analyze() raise if not overridden?
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)